# 02 - Inverse problem: finding the sources from sensor readings

*Idealised synthetic experiment. Nothing here is an estimate for any real place.*

In notebook 01 we ran the **forward** model: given how much each source emits (`q`, in micrograms per second), compute
the concentration field. Here we go **backwards**: given readings from a few sensors, estimate `q`.

**What is assumed known.** Everything except `q`: wind, diffusivity, removal rate, mixing depth, where the sources
are and how wide, the initial state and the inflow background. We only estimate the three emission rates, and they
cannot be negative.

**Key idea (linearity).** Double all emissions and the *emitted* part of the concentration doubles. So
`readings = b + A q`, where `b` is what the sensors would read with zero emissions and column `j` of `A` is what
sensor readings look like for one unit of source `j`. We compute `A` once with a few forward runs, and then fitting is
quick linear algebra. That is why we can try many sensor layouts cheaply.

In [ ]:
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

from pravah.config import load_config, scenario_from_dict, SolverOptions
from pravah.forward import simulate
from pravah.observations import SensorLayout, ObservationSpec, observe, sample
from pravah.inverse import ResponseCache, InferenceOptions, fit_sources, reconstruct_linear
from pravah.metrics import field_rmse, source_error, sensor_rmse
from pravah import plotting
%matplotlib inline

root = Path.cwd()
while not (root / "configs" / "reference.json").exists() and root != root.parent:
    root = root.parent
cfg = load_config(root / "configs" / "reference.json")
scenario = scenario_from_dict(cfg["scenario"])
solver = SolverOptions(**cfg["solver"])
q_true = np.array(cfg["q"])
OBS_TIMES = tuple(float(t) for t in np.arange(600.0, scenario.times.duration + 1, 600.0))
print("grid", scenario.domain.nx, "x", scenario.domain.ny, "| true q (ug/s):", q_true)

## 1. The truth and the sensors

We make a "true" concentration field with the forward model (this is the part we will pretend not to know), then
place sensors. Two layouts:

* **useful**: six sensors spread out *downwind* of the sources, where the plumes actually travel;
* **poor**: six sensors packed into one corner that is *upwind* of every source (the wind blows from west to east and
  slightly north), so the plumes never reach them.

In [ ]:
t0 = time.perf_counter()
truth = simulate(scenario, q_true, solver).concentration
print(f"one forward solve: {time.perf_counter() - t0:.1f} s; peak concentration {float(truth.max()):.1f} ug/m^3")

useful = SensorLayout(tuple(f"U{i}" for i in range(6)),
                      (4200.0, 5500.0, 7800.0, 5000.0, 7000.0, 8800.0),
                      (3300.0, 7200.0, 4800.0, 4500.0, 2600.0, 6500.0))
poor = SensorLayout(tuple(f"P{i}" for i in range(6)),
                    (500.0, 1000.0, 1500.0, 700.0, 1200.0, 900.0),
                    (9500.0, 9200.0, 9400.0, 8800.0, 8600.0, 9000.0))

fig = plotting.plot_sources(scenario.sources, scenario.domain, useful)
ax = fig.axes[0]
ax.plot(poor.x, poor.y, "ms", label="poor layout")
ax.legend(); ax.set_title("sources (stars), useful sensors and poor sensors")
plt.show()

## 2. Noisy observations and the fit

Each sensor reads the true concentration every 10 minutes plus random noise (standard deviation `noise_std`,
micrograms per cubic metre). `fit_sources` takes only the scenario (without `q`) and the readings; the truth is not
an input. It solves a bounded least-squares problem: choose `q >= 0` so the model readings match the data, with each
residual divided by that sensor's noise level.

We hold out one sensor from each layout: it is *not* used in the fit and serves as an honest check.

In [ ]:
cache = ResponseCache()
# The concentrations here peak at only ~0.2 ug/m^3, so we use a noise std of 0.005 ug/m^3 (a few percent of the
# peak) rather than the 0.5 stored in configs/reference.json, which would bury the signal completely.
NOISE, SEED = 0.005, int(cfg["inverse"]["seed"])

def run(layout, heldout, noise_std=NOISE, seed=SEED, alpha=0.0):
    obs = observe(truth, layout, ObservationSpec(OBS_TIMES, noise_std, seed))
    opts = InferenceOptions(method="linear", heldout_ids=(heldout,), alpha=alpha)
    fit = fit_sources(scenario, obs, opts, cache)
    return obs, fit

t0 = time.perf_counter()
obs_u, fit_u = run(useful, "U5")
print(f"first fit (includes {len(scenario.sources) + 1} forward solves for the response cache): "
      f"{time.perf_counter() - t0:.1f} s")
t0 = time.perf_counter()
obs_p, fit_p = run(poor, "P5")
print(f"second fit, other layout (cache reused): {time.perf_counter() - t0:.2f} s")

## 3. Did we recover the sources?

Compare fitted and true emission rates, and the fitted and true concentration fields.

In [ ]:
resp = cache.get(scenario, solver)
rows = []
for name, fit in (("useful", fit_u), ("poor", fit_p)):
    err = source_error(fit.q_hat, q_true)
    est = reconstruct_linear(resp, fit)
    print(f"{name:7s} q_hat = {np.round(fit.q_hat):}  rel. error (L2) = {err['relative_l2']:.3f}   "
          f"field RMSE = {field_rmse(est, truth):.3f} ug/m^3")
print("true    q     =", q_true)

fig, ax = plt.subplots(figsize=(6, 3.5))
w = 0.25; idx = np.arange(3)
ax.bar(idx - w, q_true, w, label="true"); ax.bar(idx, fit_u.q_hat, w, label="fit, useful")
ax.bar(idx + w, fit_p.q_hat, w, label="fit, poor")
ax.set_xticks(idx, [s.name for s in scenario.sources]); ax.set_ylabel("q (ug/s)"); ax.legend()
plt.show()

t_show = float(scenario.times.duration)
for name, fit in (("useful", fit_u), ("poor", fit_p)):
    fig = plotting.plot_compare_fields(truth, reconstruct_linear(resp, fit), t_show)
    fig.suptitle(f"{name} layout: truth vs fitted field vs difference at t = {t_show:.0f} s")
    plt.show()

## 4. Held-out sensors

The held-out sensors were never used in the fit. How well does the fitted field predict them (noise-free truth)?

Be careful reading this for the poor layout: its held-out sensor sits in the same empty corner as the others, where
the truth is also almost zero, so predicting "nothing" looks perfect. A held-out check only tests the places it
covers; the field comparison above (and the identifiability numbers below) show the failure.

In [ ]:
for name, layout, hid, fit in (("useful", useful, "U5", fit_u), ("poor", poor, "P5", fit_p)):
    held = layout.subset([hid])
    est = reconstruct_linear(resp, fit)
    print(f"{name:7s} held-out sensor {hid}: RMSE = {sensor_rmse(est, truth, held, OBS_TIMES):.3f} ug/m^3 "
          f"(noise std {NOISE})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=False)
for ax, (name, layout, hid, fit) in zip(axes, (("useful", useful, "U5", fit_u), ("poor", poor, "P5", fit_p))):
    held = layout.subset([hid])
    ax.plot(OBS_TIMES, sample(truth, held, OBS_TIMES).values[:, 0], "k-", label="truth")
    ax.plot(OBS_TIMES, sample(reconstruct_linear(resp, fit), held, OBS_TIMES).values[:, 0], "r--", label="fit prediction")
    ax.set_title(f"{name}: held-out {hid}"); ax.set_xlabel("time (s)"); ax.set_ylabel("ug/m^3"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Identifiability: can the data tell the sources apart?

The fitted field can look fine and still attribute emissions to the wrong source. The honest diagnostic is the
**singular values** of the (noise-whitened, scaled) matrix `A`. Read them as "how much information the sensors carry
about each combination of sources": a tiny last value means some combination of sources hardly changes any reading, so
the data cannot pin it down. The **condition number** is largest / smallest value; **rank** counts the values above
round-off. (The optimiser reporting `success` only means it converged, not that the answer is determined.)

Because the matrix is divided by the noise level, singular values well below 1 mean that combination is not resolved
at the noise level. Note the *size* of the largest value too: for the poor layout even that is astronomically small,
so the sensors see essentially nothing from any source (the rank count alone, which is relative, can still say 3).

In [ ]:
print(f"{'layout':8s} {'rank':>4s} {'condition':>11s}  singular values")
for name, fit in (("useful", fit_u), ("poor", fit_p)):
    print(f"{name:8s} {fit.rank:4d} {fit.condition:11.3g}  {np.array2string(fit.singular_values, precision=3)}"
          f"   optimiser success: {fit.success}")
fig, ax = plt.subplots(figsize=(5, 3))
for name, fit in (("useful", fit_u), ("poor", fit_p)):
    ax.semilogy(range(1, 4), np.maximum(fit.singular_values, 1e-30), "o-", label=name)
ax.axhline(1.0, color="gray", ls=":"); ax.set_xlabel("index"); ax.set_ylabel("singular value (whitened)")
ax.legend(); plt.show()

## 6. Try it yourself

Change the noise level and the random seed, then press **Run**. The sensor responses are cached, so each run takes a
fraction of a second (no new forward solves). If `ipywidgets` is not available a plain call is used instead.

In [ ]:
def report(noise_std, seed):
    for name, layout, hid in (("useful", useful, "U5"), ("poor", poor, "P5")):
        obs, fit = run(layout, hid, noise_std, seed)
        err = source_error(fit.q_hat, q_true)
        print(f"{name:7s} noise={noise_std:.2f} seed={seed}: q_hat={np.round(fit.q_hat)}  "
              f"rel. error={err['relative_l2']:.3f}  rank={fit.rank}  cond={fit.condition:.3g}")

try:
    import ipywidgets as widgets
    from IPython.display import display
    noise_w = widgets.FloatSlider(value=NOISE, min=0.0, max=2.0, step=0.1, description="noise std")
    seed_w = widgets.IntSlider(value=SEED, min=0, max=20, description="seed")
    button = widgets.Button(description="Run")
    out = widgets.Output()
    def on_click(_):
        with out:
            out.clear_output()
            report(noise_w.value, seed_w.value)
    button.on_click(on_click)  # nothing runs until the button is pressed
    display(widgets.VBox([noise_w, seed_w, button, out]))
except ImportError:
    print("ipywidgets not installed; running with noise_std=1.0, seed=2 (edit these numbers and re-run the cell).")
    report(1.0, 2)

## What to take away

* We assumed everything but `q` is known. If the wind or the diffusivity is wrong, the fit absorbs the error into `q`;
  that is a different experiment (model mismatch) and not tested here.
* With a **useful** layout the three sources are separable (large singular values, small condition number) and the
  fit recovers `q` within the noise-driven uncertainty.
* With a **poor** layout the smallest singular values collapse: the data barely depend on some sources, so their
  estimates are essentially arbitrary (often pinned at zero by the non-negativity bound). The optimiser still
  reports success.
* **An accurate concentration field is not the same as correct attribution.** Two different sets of `q` can produce
  nearly the same readings; check the identifiability diagnostics before trusting source-by-source numbers.